# Belly + Throat Identification Model — SuperPoint + LightGlue

This Colab notebook implements the identification stage used for the combined **belly + throat** crops.

**Pipeline**
1. Load the 200 colour belly + throat crops.
2. Extract SuperPoint keypoints/descriptors and match them with LightGlue.
3. Compare every unique image pair (200 × 199 / 2 = 19,900 pairs).
4. Save `all_pair_scores.csv`.
5. Rank candidates using:

`combined_score = num_matches × average_confidence`

6. Save Top-5 candidates.
7. Create Top-5 and Rank-1 LightGlue visualizations.

The combined score is a project-specific ranking score, not an identification probability or accuracy.


In [ ]:
# Install the libraries used in Colab
!pip -q install transformers accelerate opencv-python pandas matplotlib pillow


In [ ]:
# Imports
import os
import math
import itertools
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image

from transformers import AutoImageProcessor, AutoModelForKeypointMatching

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## 1. Select the belly + throat crop folder

Upload the crop folder to Colab/Google Drive and change `CROP_DIR` below.  
The images remain in **colour**; this notebook does not add a separate grayscale preprocessing stage.


In [ ]:
# OPTIONAL: mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# CHANGE THIS PATH to your folder containing the 200 belly + throat crops
CROP_DIR = Path("/content/drive/MyDrive/belly_throat_crops")

OUTPUT_DIR = Path("/content/belly_throat_identification_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

image_paths = sorted([
    p for p in CROP_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
])

print("Number of images:", len(image_paths))
print("Expected unique pairs:", len(image_paths) * (len(image_paths) - 1) // 2)

if not image_paths:
    raise FileNotFoundError(f"No images found in: {CROP_DIR}")


## 2. Load SuperPoint + LightGlue

The identification model uses the pretrained ETH-CVG LightGlue model with SuperPoint local features.


In [ ]:
MODEL_ID = "ETH-CVG/lightglue_superpoint"

processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForKeypointMatching.from_pretrained(MODEL_ID).to(DEVICE)
model.eval()

print("Loaded:", MODEL_ID)


In [ ]:
def load_rgb(path):
    return Image.open(path).convert("RGB")


def match_pair(path1, path2):
    """Match two crops and return match count, average confidence and raw result."""
    images = [load_rgb(path1), load_rgb(path2)]

    inputs = processor(images=images, return_tensors="pt")
    inputs = {k: v.to(DEVICE) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    # Convert model output to original image coordinates.
    target_sizes = [[
        (images[0].height, images[0].width),
        (images[1].height, images[1].width)
    ]]

    result = processor.post_process_keypoint_matching(
        outputs,
        target_sizes=target_sizes,
        threshold=0.0
    )[0]

    scores = result["matching_scores"]
    if torch.is_tensor(scores):
        scores = scores.detach().cpu().numpy()
    else:
        scores = np.asarray(scores)

    num_matches = int(len(scores))
    avg_confidence = float(scores.mean()) if num_matches > 0 else 0.0

    # Project-specific ranking heuristic used in the report.
    combined_score = float(num_matches * avg_confidence)

    return {
        "num_matches": num_matches,
        "average_confidence": avg_confidence,
        "combined_score": combined_score,
        "result": result
    }


## 3. Quick two-image test

This lets you confirm that the model is working before running all 19,900 comparisons.


In [ ]:
test = match_pair(image_paths[0], image_paths[1])

print("Image 1:", image_paths[0].name)
print("Image 2:", image_paths[1].name)
print("Number of matches:", test["num_matches"])
print("Average confidence:", round(test["average_confidence"], 4))
print("Combined score:", round(test["combined_score"], 4))


## 4. Compare all unique image pairs

For 200 crops this creates **19,900 unique comparisons**.  
The output is saved as `all_pair_scores.csv`.


In [ ]:
rows = []
total_pairs = len(image_paths) * (len(image_paths) - 1) // 2

for idx, (p1, p2) in enumerate(itertools.combinations(image_paths, 2), start=1):
    m = match_pair(p1, p2)

    rows.append({
        "image1": p1.name,
        "image2": p2.name,
        "num_matches": m["num_matches"],
        "average_confidence": m["average_confidence"],
        "combined_score": m["combined_score"]
    })

    if idx % 100 == 0 or idx == total_pairs:
        print(f"{idx}/{total_pairs} pairs completed")

scores_df = pd.DataFrame(rows)
scores_path = OUTPUT_DIR / "all_pair_scores.csv"
scores_df.to_csv(scores_path, index=False)

print("\nSaved:", scores_path)
print("Rows:", len(scores_df))
scores_df.head()


## 5. Build the Top-5 candidate ranking

Each stored pair is expanded in both directions so every image can act as a query.  
Candidates are sorted by the project-specific `combined_score`.


In [ ]:
directed_rows = []

for row in scores_df.itertuples(index=False):
    common = {
        "num_matches": row.num_matches,
        "average_confidence": row.average_confidence,
        "combined_score": row.combined_score
    }

    directed_rows.append({
        "query": row.image1,
        "candidate": row.image2,
        **common
    })

    directed_rows.append({
        "query": row.image2,
        "candidate": row.image1,
        **common
    })

ranking_df = pd.DataFrame(directed_rows)

ranking_df = ranking_df.sort_values(
    ["query", "combined_score", "num_matches", "average_confidence"],
    ascending=[True, False, False, False]
)

ranking_df["rank"] = ranking_df.groupby("query").cumcount() + 1

top5_df = ranking_df[ranking_df["rank"] <= 5].copy()
top5_path = OUTPUT_DIR / "top5_candidates.csv"
top5_df.to_csv(top5_path, index=False)

print("Saved:", top5_path)
print("Rows:", len(top5_df))
top5_df.head(10)


## 6. Top-5 visualization

Choose a query filename below. The notebook shows the query followed by its five highest-ranked candidates.


In [ ]:
QUERY_IMAGE = image_paths[0].name  # change if required

def image_path_from_name(name):
    return CROP_DIR / name

query_rows = top5_df[top5_df["query"] == QUERY_IMAGE].sort_values("rank")

fig, axes = plt.subplots(1, 6, figsize=(20, 5))

axes[0].imshow(load_rgb(image_path_from_name(QUERY_IMAGE)))
axes[0].set_title("Query\n" + QUERY_IMAGE, fontsize=9)
axes[0].axis("off")

for ax, row in zip(axes[1:], query_rows.itertuples(index=False)):
    ax.imshow(load_rgb(image_path_from_name(row.candidate)))
    ax.set_title(
        f"Rank {row.rank}\n{row.candidate}\n"
        f"matches={row.num_matches}\nscore={row.combined_score:.2f}",
        fontsize=8
    )
    ax.axis("off")

plt.tight_layout()
top5_vis_path = OUTPUT_DIR / "top5_visualization.png"
plt.savefig(top5_vis_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", top5_vis_path)


## 7. Rank-1 LightGlue match-line visualization

This displays the actual local correspondences found by SuperPoint + LightGlue between a query and its Rank-1 candidate.


In [ ]:
rank1_row = (
    ranking_df[(ranking_df["query"] == QUERY_IMAGE) & (ranking_df["rank"] == 1)]
    .iloc[0]
)

query_path = image_path_from_name(QUERY_IMAGE)
candidate_path = image_path_from_name(rank1_row["candidate"])

match_data = match_pair(query_path, candidate_path)
result = match_data["result"]

keypoints0 = result["keypoints0"]
keypoints1 = result["keypoints1"]
scores = result["matching_scores"]

def to_numpy(x):
    if torch.is_tensor(x):
        return x.detach().cpu().numpy()
    return np.asarray(x)

keypoints0 = to_numpy(keypoints0)
keypoints1 = to_numpy(keypoints1)
scores = to_numpy(scores)

img0 = np.array(load_rgb(query_path))
img1 = np.array(load_rgb(candidate_path))

h = max(img0.shape[0], img1.shape[0])
w0, w1 = img0.shape[1], img1.shape[1]

canvas = np.zeros((h, w0 + w1, 3), dtype=np.uint8)
canvas[:img0.shape[0], :w0] = img0
canvas[:img1.shape[0], w0:w0+w1] = img1

fig, ax = plt.subplots(figsize=(16, 8))
ax.imshow(canvas)

for p0, p1, score in zip(keypoints0, keypoints1, scores):
    x0, y0 = p0
    x1, y1 = p1
    ax.plot([x0, x1 + w0], [y0, y1], linewidth=0.6, alpha=0.55)
    ax.scatter([x0, x1 + w0], [y0, y1], s=5)

ax.set_title(
    f"{QUERY_IMAGE}  →  Rank 1: {candidate_path.name}\n"
    f"{match_data['num_matches']} matches | "
    f"average confidence = {match_data['average_confidence']:.3f}"
)
ax.axis("off")

rank1_vis_path = OUTPUT_DIR / "rank1_lightglue_matches.png"
plt.savefig(rank1_vis_path, dpi=200, bbox_inches="tight")
plt.show()

print("Saved:", rank1_vis_path)


## 8. Optional Wild-ID reference comparison

The project comparison used Wild-ID's `confirmed-matches.txt`.  
Only rows with a selected reference partner should be evaluated; `NONE` rows should not automatically be treated as true different-individual cases.

Because Wild-ID text-file formats can vary, this cell expects a simple table containing at least:
- `query`
- `reference`

Edit the column names below if your converted Wild-ID file uses different names.


In [ ]:
# OPTIONAL
# WILDID_REFERENCE_FILE = "/content/confirmed_matches.csv"
#
# wild = pd.read_csv(WILDID_REFERENCE_FILE)
# wild = wild[wild["reference"].notna()]
# wild = wild[wild["reference"].astype(str).str.upper() != "NONE"]
#
# def reference_rank(query, reference):
#     x = ranking_df[
#         (ranking_df["query"] == query) &
#         (ranking_df["candidate"] == reference)
#     ]
#     return int(x.iloc[0]["rank"]) if len(x) else np.nan
#
# wild["lightglue_rank"] = [
#     reference_rank(q, r)
#     for q, r in zip(wild["query"], wild["reference"])
# ]
#
# n = len(wild)
# top1 = (wild["lightglue_rank"] <= 1).sum()
# top5 = (wild["lightglue_rank"] <= 5).sum()
#
# print(f"Reference queries: {n}")
# print(f"Top-1: {top1}/{n} = {100*top1/n:.2f}%")
# print(f"Top-5: {top5}/{n} = {100*top5/n:.2f}%")
#
# wild.to_csv(OUTPUT_DIR / "wildid_lightglue_comparison.csv", index=False)


## 9. Download the outputs

The main identification outputs are:
- `all_pair_scores.csv`
- `top5_candidates.csv`
- `top5_visualization.png`
- `rank1_lightglue_matches.png`


In [ ]:
from google.colab import files

for filename in [
    "all_pair_scores.csv",
    "top5_candidates.csv",
    "top5_visualization.png",
    "rank1_lightglue_matches.png"
]:
    path = OUTPUT_DIR / filename
    if path.exists():
        print("Downloading:", filename)
        files.download(str(path))
